In [ ]:
# pip install yfinance pandas python-dateutil

import json
import re
from datetime import datetime, timedelta, timezone
from pathlib import Path

import pandas as pd
import yfinance as yf


TICKER = "INFY.NS"
DAYS_BACK = 30
OUT_DIR = Path("data/yfinance_infy_news")
OUT_DIR.mkdir(parents=True, exist_ok=True)

NOISE_PATTERNS = [
    r"should you buy",
    r"buy or sell",
    r"stock to buy",
    r"target price",
    r"price target",
    r"multibagger",
    r"technical pick",
    r"chart check",
    r"breakout",
    r"support level",
    r"resistance level",
    r"intraday",
    r"top \d+ stocks",
    r"stocks? to watch",
    r"nifty",
    r"sensex",
    r"brokerage",
    r"analyst rating",
    r"upgrade|downgrade",
]

SIGNAL_PATTERNS = [
    r"quarterly results|q[1-4]|earnings|revenue|profit|margin|guidance",
    r"dividend|buyback|record date",
    r"acquisition|merger|stake|subsidiary",
    r"deal|contract|client|partnership|collaboration|agreement",
    r"ceo|cfo|director|board|resignation|appointment",
    r"litigation|lawsuit|settlement|gst|tax|regulatory|sebi|sec|probe|investigation",
    r"cyber|data breach|mccamish",
    r"ai|cloud|digital transformation|finacle",
    r"esg|sustainability|brsr",
]


def text_of(item: dict) -> str:
    title = item.get("title") or ""
    publisher = item.get("publisher") or ""
    summary = item.get("summary") or ""
    return f"{title} {publisher} {summary}".lower()


def get_article_url(item: dict) -> str | None:
    # yfinance output shape can vary by version
    if item.get("link"):
        return item["link"]

    content = item.get("content") or {}
    if isinstance(content, dict):
        canonical = content.get("canonicalUrl") or {}
        if isinstance(canonical, dict) and canonical.get("url"):
            return canonical["url"]

        click = content.get("clickThroughUrl") or {}
        if isinstance(click, dict) and click.get("url"):
            return click["url"]

    return None


def get_pub_time(item: dict):
    # Older yfinance commonly uses providerPublishTime as unix seconds.
    ts = item.get("providerPublishTime")

    # Newer nested format sometimes appears.
    if ts is None and isinstance(item.get("content"), dict):
        ts = item["content"].get("pubDate")

    if isinstance(ts, int):
        return datetime.fromtimestamp(ts, tz=timezone.utc)

    if isinstance(ts, str):
        try:
            return datetime.fromisoformat(ts.replace("Z", "+00:00"))
        except ValueError:
            return None

    return None


def classify_article(item: dict) -> dict:
    text = text_of(item)

    noise_hits = [p for p in NOISE_PATTERNS if re.search(p, text, re.I)]
    signal_hits = [p for p in SIGNAL_PATTERNS if re.search(p, text, re.I)]

    if noise_hits and not signal_hits:
        return {
            "decision": "REJECT",
            "reason": "Mostly market/price/advice style article.",
            "matched_noise": noise_hits,
            "matched_signal": signal_hits,
        }

    if signal_hits:
        return {
            "decision": "ACCEPT_FOR_REVIEW",
            "reason": "Contains potentially factual Infosys/company-level signal.",
            "matched_noise": noise_hits,
            "matched_signal": signal_hits,
        }

    return {
        "decision": "NEEDS_LLM_REVIEW",
        "reason": "No obvious noise, but no strong signal from title/summary.",
        "matched_noise": noise_hits,
        "matched_signal": signal_hits,
    }


def main():
    cutoff = datetime.now(timezone.utc) - timedelta(days=DAYS_BACK)

    ticker = yf.Ticker(TICKER)
    raw_news = ticker.get_news()  # same idea as ticker.news

    records = []
    seen_urls = set()

    for item in raw_news:
        premium = (
            item.get("content", {})
            .get("finance", {})
            .get("premiumFinance", {})
        )

        if premium.get("isPremiumNews") or premium.get("isPremiumFreeNews"):
            continue
        
        published_at = get_pub_time(item)
        if published_at is None or published_at < cutoff:
            continue

        url = get_article_url(item)
        if url and url in seen_urls:
            continue
        if url:
            seen_urls.add(url)

        classification = classify_article(item)

        records.append({
            "ticker": TICKER,
            "title": item.get("title"),
            "publisher": item.get("publisher"),
            "published_at": published_at.isoformat(),
            "url": url,
            "summary": item.get("summary"),
            "decision": classification["decision"],
            "reason": classification["reason"],
            "matched_noise": classification["matched_noise"],
            "matched_signal": classification["matched_signal"],
            "raw": item,
        })

    df = pd.DataFrame(records).sort_values("published_at", ascending=False)

    csv_path = OUT_DIR / "infy_yfinance_news_last_30_days.csv"
    json_path = OUT_DIR / "infy_yfinance_news_last_30_days.json"

    df.drop(columns=["raw"], errors="ignore").to_csv(csv_path, index=False)

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False, default=str)

    print(f"Fetched recent records: {len(records)}")
    print(df["decision"].value_counts(dropna=False))
    print(f"CSV saved to: {csv_path}")
    print(f"JSON saved to: {json_path}")


if __name__ == "__main__":
    main()